md

In [ ]:
# Registered constants, read from the pre-registration at build time.
SEEDS_HERE = [1401, 1402, 1403]
STOP_SPREAD_HERE = 0.15
print('registered seeds:', SEEDS_HERE, '| stopping threshold:', STOP_SPREAD_HERE)

In [ ]:
import glob, json, os, shutil, subprocess, sys

work = '/kaggle/working/kalia'
if not os.path.exists(work):
    # Prefer the kalia-code-dev dataset. Any train.py in /kaggle/input also
    # matches a stale repo bundled inside old kernel outputs (a Sep-23
    # snapshot), which silently lacked tools/forgetting_probe.py -- the
    # same silent-wrong-source defect class as I17. Assert the source.
    hits = sorted(glob.glob('/kaggle/input/**/train.py', recursive=True))
    preferred = [p for p in hits if '/kalia-code-dev/' in p]
    assert preferred, ('kalia-code-dev is not attached or lacks train.py; '
                       'other train.py found (do NOT trust them): ' + str(hits))
    shutil.copytree(os.path.dirname(preferred[0]), work)
os.chdir(work)
for req in ('train.py', 'tools/forgetting_probe.py', 'configs/kalia-v020.yaml',
            'mixture.py', 'make_replay_shard.py'):
    assert os.path.exists(req), f'{req} missing from the copied code snapshot'
data = sorted(glob.glob('/kaggle/input/**/train.bin', recursive=True))
assert data, 'train.bin not found - attach a prep output as a kernel source'
DATA = os.path.dirname(data[0])
print('code:', work)
print('data:', DATA)
import torch
print('gpu:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('adaptive_every supported:', 'adaptive_every' in open('train.py').read())

In [ ]:
import pathlib
prev = pathlib.Path('/kaggle/working/step1_spread.json')
if not prev.exists():
    prev = pathlib.Path('/kaggle/input/step1_spread.json')
assert prev.exists(), (
    'Step 1 output not found. Run kalia-v030-baseline first: the bar for this\n     step is 1x its measured spread, not a number anyone chose.')
step1 = json.load(open(prev))
if step1['stopping']:
    raise SystemExit('Step 1 triggered its stopping rule. Do not run Step 2.')
BAR = step1['bar']
print(f"Step 1 spread: {BAR:.4f} nats -> this step's bar is delta <= {-BAR:.4f}")

In [ ]:
arms = ['v030-norewarm-s1401', 'v030-rewarm-s1401']
for a in arms:
    assert os.path.exists(f'configs/{a}.yaml'), f'missing configs/{a}.yaml'
print('arms:', arms)
ckpt_hits = sorted(glob.glob('/kaggle/input/**/ckpt.pt', recursive=True))
RESUME_CKPT = ckpt_hits[0] if ckpt_hits else 'kalia-lm/kalia-v020'
print('resume checkpoint:', RESUME_CKPT)
print(subprocess.run([sys.executable, 'ablate.py', '--arms', ','.join(arms),
                       '--steps', '500', '--data-dir', DATA,
                       '--resume-from', RESUME_CKPT,
                       '--out-root', '/kaggle/working/out'],
                      capture_output=True, text=True).stdout[-5000:])

In [ ]:
import pandas as pd
losses = {}
for a in arms:
    df = pd.read_csv(f'/kaggle/working/out/{a}/val_log.csv')
    print(f'{a:26s} ' + '  '.join(f"{int(r.step)}={r.val_loss:.4f}" for r in df.itertuples()))
    losses[a] = float(df.iloc[-1].val_loss)
delta = losses['v030-rewarm-s1401'] - losses['v030-norewarm-s1401']
passes = delta <= -BAR
print(f'\nre-warm vs no re-warm: {delta:+.4f} nats | bar <= {-BAR:.4f}')
print('TRANSFERS' if passes else 'DOES NOT TRANSFER at 58M')
if not passes:
    print('  Report the negative. Do not tune the schedule until it passes --')
    print('  that is searching for a result, and Step 3 is registered as the')
    print('  measurement that would justify any tuning.')
json.dump({'losses': losses, 'delta': delta, 'bar': BAR, 'passes': passes},
          open('/kaggle/working/step2_rewarm.json', 'w'), indent=2)